# Employee Dimension

This auxiliary notebook consolidates every employee identifier referenced by E-SMAT inspections and prepares a governed employee dimension for analytical use.

## Why this notebook is separate

Employee consolidation is not part of the core inspection fact model. It is an auxiliary process that:

1. Reads employee identifiers from the inspection header;
2. Converts the observer and observed-person columns into role-level rows;
3. Removes empty and duplicate identifiers;
4. Preserves the inspection, site, shift, and employee-role context;
5. Optionally enriches identifiers from an authorized employee reference table;
6. Publishes an employee dimension and an inspection-employee bridge.

## Outputs

- `employee_dimension`
- `inspection_employee_bridge`

The bridge is recommended because one inspection can reference several employees in different roles. The employee dimension contains one row per employee identifier.

> The public version does not contain employee names, corporate directory data, or real identifiers. Enrichment must use an authorized private source.


## 1. Load shared functions

In [ ]:
%run ./01_shared_functions


## 2. Configure sources and targets

`EMPLOYEE_MASTER_TABLE` is optional. Leave it as `None` until an authorized employee reference source is available.


In [ ]:
GOLD_CATALOG = "<catalog>"
GOLD_SCHEMA = "<gold-schema>"
AUXILIARY_CATALOG = "<catalog>"
AUXILIARY_SCHEMA = "<auxiliary-schema>"

INSPECTION_HEADER_TABLE = (
    f"{GOLD_CATALOG}.{GOLD_SCHEMA}.inspection_header"
)

EMPLOYEE_MASTER_TABLE = None
# Example private value:
# EMPLOYEE_MASTER_TABLE = "<catalog>.<schema>.<authorized-employee-table>"

TARGET_TABLES = {
    "employee_dimension": (
        f"{AUXILIARY_CATALOG}.{AUXILIARY_SCHEMA}.employee_dimension"
    ),
    "inspection_employee_bridge": (
        f"{AUXILIARY_CATALOG}.{AUXILIARY_SCHEMA}.inspection_employee_bridge"
    ),
}

WRITE_MODE = "overwrite"
OVERWRITE_SCHEMA = "true"


## 3. Validate configuration

In [ ]:
def validate_employee_dimension_config(
    inspection_header_table,
    employee_master_table,
    target_tables,
):
    if not inspection_header_table or inspection_header_table.count(".") != 2:
        raise ValueError(
            f"Invalid inspection-header table: {inspection_header_table}"
        )

    if employee_master_table is not None and employee_master_table.count(".") != 2:
        raise ValueError(
            f"Invalid employee master table: {employee_master_table}"
        )

    required_targets = {
        "employee_dimension",
        "inspection_employee_bridge",
    }
    missing_targets = sorted(required_targets - set(target_tables))

    if missing_targets:
        raise ValueError(
            f"Missing auxiliary target configuration: {missing_targets}"
        )

    for dataset_name, table_name in target_tables.items():
        if not table_name or table_name.count(".") != 2:
            raise ValueError(
                f"Invalid target table for '{dataset_name}': {table_name}"
            )

    if len(target_tables.values()) != len(set(target_tables.values())):
        raise ValueError("Auxiliary target tables must be unique.")

    print("Employee-dimension configuration validated successfully.")


validate_employee_dimension_config(
    INSPECTION_HEADER_TABLE,
    EMPLOYEE_MASTER_TABLE,
    TARGET_TABLES,
)


## 4. Read the Gold inspection header

In [ ]:
if not spark.catalog.tableExists(INSPECTION_HEADER_TABLE):
    raise RuntimeError(
        "Gold inspection-header table was not found. "
        "Run 05_delta_load.ipynb before this auxiliary notebook."
    )

header_df = spark.table(INSPECTION_HEADER_TABLE)

require_columns(
    dataframe=header_df,
    required_columns={
        "inspection_id",
        "inspection_date",
        "site",
        "shift",
        "observer_id",
        "observed_person_id_1",
        "observed_person_id_2",
        "observed_person_id_3",
    },
    dataset_name="inspection_header_gold",
)

print("Gold inspection-header dataset loaded successfully.")


## 5. Convert employee columns into role-level rows

The four employee columns are converted into a normalized structure. This avoids forcing one employee dimension to maintain several competing active relationships with the inspection header.

```text
observer_id           → observer
observed_person_id_1  → observed_1
observed_person_id_2  → observed_2
observed_person_id_3  → observed_3
```


In [ ]:
employee_role_mapping = {
    "observer_id": "observer",
    "observed_person_id_1": "observed_1",
    "observed_person_id_2": "observed_2",
    "observed_person_id_3": "observed_3",
}

role_dataframes = []

for source_column, employee_role in employee_role_mapping.items():
    role_dataframe = header_df.select(
        "inspection_id",
        "inspection_date",
        "site",
        "shift",
        F.trim(F.col(source_column).cast("string")).alias("employee_id"),
        F.lit(employee_role).alias("employee_role"),
        F.lit(source_column).alias("source_column"),
    )

    role_dataframes.append(role_dataframe)

inspection_employee_bridge_df = role_dataframes[0]

for role_dataframe in role_dataframes[1:]:
    inspection_employee_bridge_df = (
        inspection_employee_bridge_df.unionByName(role_dataframe)
    )

inspection_employee_bridge_df = (
    inspection_employee_bridge_df
    .filter(
        F.col("employee_id").isNotNull()
        & (F.col("employee_id") != "")
    )
    .withColumn(
        "inspection_employee_key",
        F.concat_ws(
            "_",
            F.col("inspection_id"),
            F.col("employee_role"),
            F.col("employee_id"),
        ),
    )
    .dropDuplicates(["inspection_employee_key"])
)

print("Inspection employee roles normalized successfully.")


## 6. Build the base employee dimension

The base dimension contains one row per employee identifier. Site and shift are derived deterministically from the most recent inspection context available for that employee.


In [ ]:
from pyspark.sql.window import Window

employee_context_window = Window.partitionBy("employee_id").orderBy(
    F.col("inspection_date").desc_nulls_last(),
    F.col("inspection_id").desc(),
)

latest_employee_context_df = (
    inspection_employee_bridge_df
    .withColumn(
        "context_rank",
        F.row_number().over(employee_context_window),
    )
    .filter(F.col("context_rank") == 1)
    .select(
        "employee_id",
        F.col("site").alias("latest_site"),
        F.col("shift").alias("latest_shift"),
        F.col("inspection_date").alias("latest_inspection_date"),
    )
)

employee_activity_df = (
    inspection_employee_bridge_df
    .groupBy("employee_id")
    .agg(
        F.countDistinct("inspection_id").alias("inspection_count"),
        F.min("inspection_date").alias("first_inspection_date"),
        F.max("inspection_date").alias("last_inspection_date"),
        F.sort_array(
            F.collect_set("employee_role")
        ).alias("employee_roles"),
    )
)

employee_dimension_base_df = (
    employee_activity_df
    .join(
        latest_employee_context_df,
        on="employee_id",
        how="left",
    )
)

print("Base employee dimension created successfully.")


## 7. Optionally enrich from an authorized employee source

The private employee source should contain one row per employee identifier. The notebook expects `employee_id` and can preserve selected attributes when they are available.

The public notebook does not fabricate production names. If no private source is configured, `employee_name` and other directory attributes remain null.


In [ ]:
EMPLOYEE_MASTER_COLUMN_MAPPING = {
    "employee_id": "employee_id",
    "employee_name": "employee_name",
    "employee_status": "employee_status",
    "job_title": "employee_job_title",
    "department": "employee_department",
}

if EMPLOYEE_MASTER_TABLE is not None:
    if not spark.catalog.tableExists(EMPLOYEE_MASTER_TABLE):
        raise RuntimeError(
            f"Authorized employee master table not found: "
            f"{EMPLOYEE_MASTER_TABLE}"
        )

    employee_master_source_df = spark.table(EMPLOYEE_MASTER_TABLE)

    source_employee_id = EMPLOYEE_MASTER_COLUMN_MAPPING["employee_id"]

    require_columns(
        dataframe=employee_master_source_df,
        required_columns={source_employee_id},
        dataset_name="authorized_employee_master",
    )

    available_master_columns = [
        source_column
        for source_column in EMPLOYEE_MASTER_COLUMN_MAPPING.values()
        if source_column in employee_master_source_df.columns
    ]

    employee_master_df = employee_master_source_df.select(
        *[
            F.col(source_column).alias(target_column)
            for target_column, source_column
            in EMPLOYEE_MASTER_COLUMN_MAPPING.items()
            if source_column in available_master_columns
        ]
    )

    employee_master_df = (
        employee_master_df
        .withColumn(
            "employee_id",
            F.trim(F.col("employee_id").cast("string")),
        )
        .dropDuplicates(["employee_id"])
    )

    employee_dimension_df = employee_dimension_base_df.join(
        employee_master_df,
        on="employee_id",
        how="left",
    )

    enrichment_source = EMPLOYEE_MASTER_TABLE
else:
    employee_dimension_df = (
        employee_dimension_base_df
        .withColumn("employee_name", F.lit(None).cast("string"))
        .withColumn("employee_status", F.lit(None).cast("string"))
        .withColumn("employee_job_title", F.lit(None).cast("string"))
        .withColumn("employee_department", F.lit(None).cast("string"))
    )

    enrichment_source = None

print(
    "Employee enrichment completed. "
    f"Source: {enrichment_source or 'not configured'}"
)


## 8. Add dimension attributes and stable ordering

`employee_sort_order` provides a deterministic numeric order for Power BI. The order does not expose or infer a corporate hierarchy.


In [ ]:
employee_sort_window = Window.orderBy(
    F.col("employee_id").asc()
)

employee_dimension_df = (
    employee_dimension_df
    .withColumn(
        "employee_sort_order",
        F.row_number().over(employee_sort_window),
    )
    .withColumn(
        "employee_display_name",
        F.when(
            F.col("employee_name").isNotNull()
            & (F.trim(F.col("employee_name")) != ""),
            F.concat_ws(
                " - ",
                F.col("employee_id"),
                F.col("employee_name"),
            ),
        ).otherwise(F.col("employee_id")),
    )
    .withColumn(
        "employee_roles_text",
        F.concat_ws(", ", F.col("employee_roles")),
    )
    .withColumn(
        "enrichment_status",
        F.when(
            F.col("employee_name").isNotNull()
            & (F.trim(F.col("employee_name")) != ""),
            F.lit("enriched"),
        ).otherwise(F.lit("not_enriched")),
    )
    .withColumn("published_timestamp", F.current_timestamp())
)

inspection_employee_bridge_df = (
    inspection_employee_bridge_df
    .withColumn("published_timestamp", F.current_timestamp())
)

print("Employee analytical attributes created successfully.")


## 9. Validate the employee dimension and bridge

In [ ]:
require_columns(
    dataframe=employee_dimension_df,
    required_columns={
        "employee_id",
        "employee_display_name",
        "employee_sort_order",
        "inspection_count",
        "latest_site",
        "latest_shift",
        "employee_roles_text",
        "enrichment_status",
        "published_timestamp",
    },
    dataset_name="employee_dimension",
)

require_columns(
    dataframe=inspection_employee_bridge_df,
    required_columns={
        "inspection_employee_key",
        "inspection_id",
        "employee_id",
        "employee_role",
        "site",
        "shift",
        "inspection_date",
        "published_timestamp",
    },
    dataset_name="inspection_employee_bridge",
)

duplicate_employee_count = (
    employee_dimension_df.groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_employee_count > 0:
    raise ValueError(
        "Employee dimension contains duplicate employee_id values."
    )

duplicate_bridge_key_count = (
    inspection_employee_bridge_df
    .groupBy("inspection_employee_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_bridge_key_count > 0:
    raise ValueError(
        "Inspection-employee bridge contains duplicate keys."
    )

invalid_role_count = inspection_employee_bridge_df.filter(
    ~F.col("employee_role").isin(
        "observer",
        "observed_1",
        "observed_2",
        "observed_3",
    )
).count()

if invalid_role_count > 0:
    raise ValueError(
        "Inspection-employee bridge contains an invalid employee role."
    )

orphan_bridge_count = inspection_employee_bridge_df.join(
    employee_dimension_df.select("employee_id"),
    on="employee_id",
    how="left_anti",
).count()

if orphan_bridge_count > 0:
    raise ValueError(
        "Inspection-employee bridge contains identifiers missing from the employee dimension."
    )

employee_count = employee_dimension_df.count()
bridge_count = inspection_employee_bridge_df.count()

print(
    f"Employee model validated: {employee_count} employees and "
    f"{bridge_count} inspection-role records."
)


## 10. Persist auxiliary Delta tables

In [ ]:
auxiliary_dataframes = {
    "employee_dimension": employee_dimension_df,
    "inspection_employee_bridge": inspection_employee_bridge_df,
}

load_summary = []

for dataset_name, dataframe in auxiliary_dataframes.items():
    target_table = TARGET_TABLES[dataset_name]
    record_count = dataframe.count()

    print(f"Writing auxiliary dataset '{dataset_name}' to {target_table}...")

    (
        dataframe.write.format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", OVERWRITE_SCHEMA)
        .saveAsTable(target_table)
    )

    load_summary.append(
        {
            "dataset": dataset_name,
            "target_table": target_table,
            "records": record_count,
            "status": "written",
        }
    )

    print(f"Auxiliary table written successfully: {record_count} records.")

load_summary_df = spark.createDataFrame(load_summary)
display(load_summary_df.orderBy("dataset"))

print("Employee dimension publishing completed successfully.")


## 11. Power BI model usage

Recommended relationships:

```text
employee_dimension[employee_id]
        1
        │
        *
inspection_employee_bridge[employee_id]
        *
        │
        1
inspection_header[inspection_id]
```

This bridge pattern allows one employee dimension to filter observer and observed-person roles without maintaining four competing relationships to the inspection header.

For compatibility with an existing report that expects `d_lista_matriculas`, the `employee_dimension` table can be renamed or mapped in Power Query. Suggested field mapping:

```text
employee_id           → Matricula
employee_name         → NomeFuncionario
latest_site           → Planta
latest_shift          → Turno
employee_sort_order   → OrdemMatricula
```

Use `employee_sort_order` as the sort-by column for employee identifiers or display labels.


## Privacy note

Employee names and directory attributes must come only from an authorized private source and must not be committed to the public repository. The public Power BI mock should use clearly synthetic names and identifiers.
